# Ingestion and Labeling Notebook

TODO: Add notebook introduction

## Environment Setup

TODO: Briefly outline environment setup

In [1]:
# Import libraries

# Standard Library
import sys
from pathlib import Path
import pandas as pd
from collections import Counter

# Repository Root Setup
p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))

# Custom Modules
from src.config import load_settings
from src.spark.session import get_spark
from src.spark.ingestion.spark.transforms.recipes_spark import clean_recipes
from src.spark.ingestion.spark.transforms.interactions_spark import clean_interactions
from src.spark.ingestion.spark.transforms.merge_spark import build_gold_reviews
from src.spark.labeling.zero_shot import (
    ZeroShotConfig, 
    prepare_zero_shot_input_from_gold, 
    run_zero_shot_incremental_with_checkpoints,
    attach_zero_shot_labels_to_gold,
    run_final_label_report
)

# Initialize Settings and Spark
s = load_settings(prefer_latest_run=True)
spark = get_spark(app_name="01_ingestion_and_labeling", debug=True)

print(f"Environment Ready: {s.env}")
print(f"Processed Dir: {s.processed_dir}")

# Spark
import pyspark.sql.functions as F

your 131072x1 screen size is bogus. expect trouble
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 23:49:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/06 23:49:16 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/06 23:49:16 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Environment Ready: local
Processed Dir: /home/iauger/projects/dsci632-project/data/processed


## Ingestion Logic

TODO: Describe process of downloading RAW and processing to Bronze and then to Silver

In [2]:
# Read Raw CSVs
recipes_raw = spark.read.option("header", "true").option("multiline", "true").csv(s.raw_recipes_path)
interactions_raw = spark.read.option("header", "true").option("multiline", "true").csv(s.raw_interactions_path)

# Clean and Move to Silver
recipes_s = clean_recipes(recipes_raw)
interactions_s = clean_interactions(interactions_raw)

# Create the Base 'Gold' Reviews (includes the review_key hash)
gold_reviews = build_gold_reviews(interactions_s)

# Save to Disk
gold_reviews.write.mode("overwrite").parquet(s.gold_reviews_path)
print(f"Silver/Gold Ingestion Complete: {gold_reviews.count():,} reviews.")

Silver/Gold Ingestion Complete: 931,407 reviews.


In [3]:
# Block 2.1: Pre-Labeling Data Quality Check
print("--- Silver Review Quality Metrics ---")
# 1. Check for Nulls/Empties that break transformers
gold_reviews = spark.read.parquet(s.gold_reviews_path)
gold_reviews.select(
    F.sum(F.col("review_clean").isNull().cast("int")).alias("null_reviews"),
    F.sum((F.length(F.trim(F.col("review_clean"))) == 0).cast("int")).alias("empty_strings")
).show()

# 2. Token Length Distribution
# This confirms if our min_tokens=15 rule is too restrictive
review_stats = gold_reviews.withColumn("tokens", F.size(F.split(F.col("review_clean"), r"\s+")))
print("--- Review Token Percentiles ---")
review_stats.selectExpr("percentile_approx(tokens, array(0.1, 0.25, 0.5, 0.75, 0.9)) as p").show()

--- Silver Review Quality Metrics ---
+------------+-------------+
|null_reviews|empty_strings|
+------------+-------------+
|           0|            0|
+------------+-------------+

--- Review Token Percentiles ---


+--------------------+
|                   p|
+--------------------+
|[22, 31, 46, 65, 86]|
+--------------------+



## Zero-Shot Labeling

TODO: Outline the why here, the approach, and how labels are being stored/updated

In [22]:
from pyspark.sql import functions as F

# 1. Define the Sinkhole Keywords
sinkhole_keywords = {
    "too_spicy": ["spicy", "hot", "cayenne", "chili", "burn", "heat", "kick"],
    "too_salty": ["salty", "salt", "sodium", "brine"],
    "too_acidic": ["acidic", "vinegar", "lemon", "lime", "tart", "sour", "tangy"],
    "too_sweet": ["sweet", "sugar", "syrup", "cloying", "honey"],
    "mushy_soggy": ["mushy", "soggy", "wet", "limp", "texture"],
    "bland_lacks_flavor": ["bland", "tasteless", "flavorless", "boring", "plain"]
}

all_keywords = [kw for sublist in sinkhole_keywords.values() for kw in sublist]
regex_pattern = "|".join([rf"\b{kw}\b" for kw in all_keywords])

# 2. Load and Filter out existing labels
gold_reviews = spark.read.parquet(s.gold_reviews_path)
v7_path = Path(s.processed_dir) / "labeling" / "zero_shot" / "labeled_gold_reviews_v7.parquet"

if v7_path.exists():
    labeled_keys = spark.read.parquet(str(v7_path)).select("review_key").distinct()
    gold_reviews = gold_reviews.join(labeled_keys, on="review_key", how="left_anti")

# 3. Apply the "Conflict + Keyword" logic
# We want 4+ star reviews that CONTAIN these specific "risk" keywords
conflict_candidates = gold_reviews.filter(
    (F.col("rating") >= 4) & 
    (F.col("review_clean").rlike(regex_pattern))
)

# 4. Sample to hit your 100k target (adjust 40k as needed)
target_n = 40000
current_pool = conflict_candidates.count()

if current_pool < target_n:
    print(f"Only found {current_pool:,} conflict rows. Sampling all of them.")
    final_sample = conflict_candidates
else:
    final_sample = conflict_candidates.sample(False, target_n / current_pool, seed=42).limit(target_n)

# 5. Write to disk
sample_out_path = Path(s.gold_dir) / "labeling_sample_v8_conflict.parquet"
final_sample.write.mode("overwrite").parquet(str(sample_out_path))

print(f"Prepared {final_sample.count():,} Conflict + Keyword rows for final labeling.")

Prepared 40,000 Conflict + Keyword rows for final labeling.


In [23]:
# Set to true to run on the keyword-targeted sample instead of the full gold set
ALTERNATE_SAMPLE = True

# Configure Zero-Shot Labeling
cfg = ZeroShotConfig(
    taxonomy_version="v1",
    min_tokens=15,
    sample_n=40000,   
    batch_size=4,     
    sample_seed=42
)

if ALTERNATE_SAMPLE:
    print("Running on the alternate keyword-targeted sample for zero-shot labeling.")
    model_path = str(sample_out_path)
else:
    print("Running on the full gold set for zero-shot labeling.")
    model_path = str(s.gold_reviews_path)
    
# Prepare/Check Input
inp_path = prepare_zero_shot_input_from_gold(spark, model_path, cfg, s.processed_dir)

# Run Incremental 
print("Starting incremental zero-shot run...")
out_path = run_zero_shot_incremental_with_checkpoints(inp_path, cfg, s.processed_dir)

Running on the alternate keyword-targeted sample for zero-shot labeling.


Starting incremental zero-shot run...


Loading weights:   0%|          | 0/231 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Zero-shot batches:   0%|          | 0/125 [00:00<?, ?batch/s]

...

Zero-shot batches:   0%|          | 0/125 [00:00<?, ?batch/s]

Loading weights:   0%|          | 0/231 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Zero-shot batches:   0%|          | 0/125 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [24]:
# Sync to the final Gold Labeled file
labeled_path = attach_zero_shot_labels_to_gold(
    spark, 
    model_path, 
    s.processed_dir, 
    cfg, 
    out_path=f"{s.processed_dir}/labeling/zero_shot/labeled_gold_reviews_v8.parquet"
)

run_final_label_report(labeled_path)

--- Final Labeling Report: labeled_gold_reviews_v8.parquet ---
Total Labeled Samples: 2,000
Avg Labels/Review: 3.26

Top Tags (% of dataset):
delicious_tasty............... 1,617 (80.85%)
would_make_again.............. 1,537 (76.85%)
family_hit.................... 1,289 (64.45%)
substitution_modification..... 1,110 (55.50%)
easy_quick.................... 547 (27.35%)
ingredient_issue.............. 120 (6.00%)
moist_tender.................. 60 (3.00%)
would_not_make_again.......... 58 (2.90%)
time_consuming_complex........ 55 (2.75%)
bland_lacks_flavor............ 53 (2.65%)

Reviews with 0 labels: 119 (5.95%)


In [25]:
from pyspark.sql import functions as F

# 1. Load both sets
v7_path = f"{s.processed_dir}/labeling/zero_shot/labeled_gold_reviews_v7.parquet"
v8_path = f"{s.processed_dir}/labeling/zero_shot/labeled_gold_reviews_v8.parquet"

df_v7 = spark.read.parquet(v7_path)
df_v8 = spark.read.parquet(v8_path)

# 2. Combine and Deduplicate (just in case some 10k rows overlapped)
# 'review_key' is your primary unique identifier
combined_labeled_df = df_v7.unionByName(df_v8).dropDuplicates(["review_key"])

# 3. Overwrite v8 to establish the new 'Master' labeled set
combined_labeled_df.repartition(20).write.mode("overwrite").parquet(v8_path)

run_final_label_report(v8_path)

--- Final Labeling Report: labeled_gold_reviews_v8.parquet ---
Total Labeled Samples: 64,946
Avg Labels/Review: 3.22

Top Tags (% of dataset):
delicious_tasty............... 49,959 (76.92%)
would_make_again.............. 48,411 (74.54%)
family_hit.................... 42,104 (64.83%)
substitution_modification..... 31,066 (47.83%)
easy_quick.................... 18,848 (29.02%)
would_not_make_again.......... 4,822 (7.42%)
ingredient_issue.............. 4,579 (7.05%)
bland_lacks_flavor............ 3,000 (4.62%)
time_consuming_complex........ 2,031 (3.13%)
moist_tender.................. 1,796 (2.77%)

Reviews with 0 labels: 3967 (6.11%)


## Conclusion

TODO: Review the full ingestion to labeling pipeline and how this sets up the upcoming stages. 